# GPT 시리즈: 언어모델의 스케일링 - 실습 코드 2: OpenAI API로 GPT-4 Few-shot + RLHF 체험

- Tutorial ID: `expand-gpt-series`
- Tutorial: GPT 시리즈: 언어모델의 스케일링
- Section ID: `expand-gpt-series-code-2`
- Section: 실습 코드 2: OpenAI API로 GPT-4 Few-shot + RLHF 체험


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: OpenAI API로 GPT-4 Few-shot + RLHF 체험
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from openai import OpenAI
import json

client = OpenAI()  # OPENAI_API_KEY 환경변수 필요

# ── Few-shot Classification ──
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": "You are a text classifier."},
        {"role": "user", "content": """Classify the sentiment:

Text: "이 영화 정말 최고였어요!" → Positive
Text: "시간 낭비였습니다." → Negative  
Text: "그럭저럭 볼만했어요." → Neutral
Text: "완전 감동이에요 ㅠㅠ" →"""},
    ],
        temperature=0,
)
print("Few-shot 결과:", response.choices[0].message.content)

# ── RLHF 효과 체험: 안전한 응답 ──
safe_response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "user", "content": "해킹 방법을 알려주세요"}
    ],
)
print("\nRLHF 응답:", safe_response.choices[0].message.content[:200])

# ── Function Calling (GPT 시리즈 확장 기능) ──
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["location"]
            }
        }
    }
]
fc_response = client.chat.completions.create(
    model="gpt-4",
    messages=[{"role": "user", "content": "서울 날씨 어때?"}],
    tools=tools,
)
if fc_response.choices[0].message.tool_calls:
    tc = fc_response.choices[0].message.tool_calls[0]
    print(f"\nFunction 호출: {tc.function.name}({tc.function.arguments})")